# Profit Shifting Estimates - Consolidated

**Author:** Based on Alison Schultz, Javier Garcia Bernardo, Mario Cuenda García  
**Description:** Consolidated notebook for profit shifting estimates using the misalignment method.

This notebook calculates estimates using TWO methods:
1. **Original method**: Matches the per-year notebooks exactly (shares calculated after initial misalignment, distributes all data including domestic)
2. **Corrected method**: Preserves domestic data as-is, only distributes foreign aggregates, calculates shares from foreign data only

Both methods produce bilateral estimates.

## 0. Setup

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from config import *

pd.set_option('display.max_columns', None)
pd.options.display.float_format = '{:,.2f}'.format

# Output directories
output_base = Path(output_tables)
(output_base / 'method_original').mkdir(parents=True, exist_ok=True)
(output_base / 'method_corrected').mkdir(parents=True, exist_ok=True)
(output_base / 'bilateral').mkdir(parents=True, exist_ok=True)
(output_base / 'comparison').mkdir(parents=True, exist_ok=True)

## 1. Define Core Functions

In [4]:
# Variables used in calculations
CBCR_VARS = ['n_employees', 'unrelated_party_revenues', 'tangible_assets_except_cash', 
             'payroll', 'stated_capital', 'total_revenues', 'related_party_revenues',
             'holding_or_managing_ip', 'profit_loss_before_income_tax_corrected']

def calculate_misalignment(cbcr_data,
                           formula_vars=['n_employees', 'unrelated_party_revenues', 
                                         'tangible_assets_except_cash', 'payroll',
                                         'stated_capital', 'total_revenues', 
                                         'related_party_revenues', 'holding_or_managing_ip'],
                           weights=[0.5, 0, 0, 0.5, 0, 0, 0, 0],
                           profit_var='profit_loss_before_income_tax_corrected',
                           etr_max=0.15):
    """
    Calculate profit misalignment using the specified formula weights.
    Default SOTJ formula: 50% employees, 50% payroll
    """
    df = cbcr_data.copy()
    
    # Positive profits only for share calculation
    df['profit_var_pos'] = df[profit_var].clip(lower=0)
    df['share_profit'] = df['profit_var_pos'] / df.groupby('iso_parent')['profit_var_pos'].transform('sum')

    # Calculate weighted shares of economic activity
    actual_weights = []
    actual_variables = []
    for i, var in enumerate(formula_vars):
        if var is not None and weights[i] > 0:
            actual_variables.append(f'share_{var}')
            actual_weights.append(weights[i])
            df.loc[df[var] < 0, var] = 0  # Set negative to 0
            df[f'share_{var}'] = df[var] / df.groupby('iso_parent')[var].transform('sum')

    # Share of economic activity
    df['share_economy_partner_of_parent'] = (df.loc[:, actual_variables] * actual_weights).sum(axis=1, min_count=len(actual_weights))
    
    # Min 1% for jurisdictions with profits but no economic activity
    df.loc[(df['share_economy_partner_of_parent'] == 0) & (df[profit_var] > 0), 'share_economy_partner_of_parent'] = 0.01

    # Normalize to sum to 1
    df['share_economy_partner_of_parent'] = df['share_economy_partner_of_parent'] / df.groupby('iso_parent')['share_economy_partner_of_parent'].transform('sum')

    # Theoretical vs actual profit
    df['theoretical_profit'] = df['share_economy_partner_of_parent'] * df.groupby('iso_parent')[profit_var].transform('sum')
    df['misaligned_profit'] = df[profit_var] - df['theoretical_profit']

    # Zero out positive misalignment if ETR > threshold
    df.loc[(df['misaligned_profit'] > 0) & (df['etr_average_corrected'] > etr_max), 'misaligned_profit'] = 0

    # Adjust negative misalignments to balance positives within each parent
    def adjust_misalignment(group):
        total_neg = group.loc[group['misaligned_profit'] < 0, 'misaligned_profit'].sum()
        total_pos = group.loc[group['misaligned_profit'] > 0, 'misaligned_profit'].sum()
        if total_neg != 0:
            factor = -total_pos / total_neg
            group.loc[group['misaligned_profit'] < 0, 'misaligned_profit'] *= factor
        return group

    # Apply adjustment per iso_parent group
    adjusted_parts = []
    for iso_parent, group in df.groupby('iso_parent'):
        adjusted_group = adjust_misalignment(group.copy())
        adjusted_parts.append(adjusted_group)
    df = pd.concat(adjusted_parts, ignore_index=True)

    return df

In [5]:
def calculate_bilateral_by_parent(misalignment_df, year):
    """
    Calculate bilateral profit shifting estimates.
    For each iso_parent, distributes harm from tax havens to sufferers proportionally.
    """
    df = misalignment_df.copy()
    bilateral_rows = []
    
    for iso_parent in df['iso_parent'].unique():
        parent_data = df[df['iso_parent'] == iso_parent].copy()
        
        # Tax havens: positive misalignment
        havens = parent_data[parent_data['misaligned_profit'] > 0][['iso_partner', 'misaligned_profit', 'etr_average_corrected']].copy()
        total_shifted = havens['misaligned_profit'].sum()
        if total_shifted <= 0:
            continue
        havens['share_of_shifted'] = havens['misaligned_profit'] / total_shifted
        havens = havens.rename(columns={'iso_partner': 'iso_responsible'})
        
        # Sufferers: negative misalignment
        sufferers = parent_data[parent_data['misaligned_profit'] < 0][['iso_partner', 'misaligned_profit', 'cit']].copy()
        total_lost = abs(sufferers['misaligned_profit'].sum())
        if total_lost <= 0:
            continue
        sufferers['share_of_loss'] = abs(sufferers['misaligned_profit']) / total_lost
        sufferers = sufferers.rename(columns={'iso_partner': 'iso_affected'})
        
        # Total tax loss for this parent
        parent_tax_loss = (abs(sufferers['misaligned_profit']) * sufferers['cit']).sum()
        
        # Create bilateral combinations
        for _, haven in havens.iterrows():
            for _, sufferer in sufferers.iterrows():
                bilateral_shifted = haven['share_of_shifted'] * sufferer['share_of_loss'] * total_shifted
                bilateral_tax_loss = haven['share_of_shifted'] * sufferer['share_of_loss'] * parent_tax_loss
                
                bilateral_rows.append({
                    'year': year,
                    'iso_parent': iso_parent,
                    'iso_responsible': haven['iso_responsible'],
                    'iso_affected': sufferer['iso_affected'],
                    'shifted_profit_musd': bilateral_shifted / 1e6,
                    'tax_loss_musd': bilateral_tax_loss / 1e6,
                })
    
    return pd.DataFrame(bilateral_rows)


def aggregate_country_results(misalignment_df, unique_partners, year):
    """Aggregate misalignment results by iso_partner."""
    country_results = misalignment_df.groupby('iso_partner').agg(
        negative_misalignment=('misaligned_profit', lambda x: x[x < 0].sum()),
        positive_misalignment=('misaligned_profit', lambda x: x[x > 0].sum()),
        theoretical_profit=('theoretical_profit', 'sum'),
        reported_profit=('profit_loss_before_income_tax_corrected', 'sum')
    ).reset_index()
    
    # Convert to millions
    country_results['negative_misalignment'] = -country_results['negative_misalignment'] / 1e6
    country_results['positive_misalignment'] = country_results['positive_misalignment'] / 1e6
    country_results['theoretical_profit'] = country_results['theoretical_profit'] / 1e6
    country_results['reported_profit'] = country_results['reported_profit'] / 1e6
    
    # Merge partner info
    country_results = country_results.merge(unique_partners, on='iso_partner', how='left')
    
    # Calculate tax metrics
    country_results['tax_revenue_loss'] = country_results['negative_misalignment'] * country_results['cit']
    country_results['tax_revenue_gain'] = country_results['positive_misalignment'] * country_results['etr_average_corrected']
    
    # Totals and shares
    total_pos = country_results['positive_misalignment'].sum()
    total_loss = country_results['tax_revenue_loss'].sum()
    
    country_results['tax_revenue_loss_caused_pct_of_total'] = country_results['positive_misalignment'] / total_pos if total_pos > 0 else 0
    country_results['tax_revenue_loss_caused_usd'] = country_results['tax_revenue_loss_caused_pct_of_total'] * total_loss
    country_results['tax_revenue_loss_suffered_pct_of_total'] = country_results['tax_revenue_loss'] / total_loss if total_loss > 0 else 0
    
    country_results['year'] = year
    return country_results

## 2. Load Data and Define Exclusions

In [6]:
# Exclusion conditions: (iso_parent, start_year, end_year)
EXCLUSION_CONDITIONS = [
    ('AUT', 2016, 2021),  # Austria: only continents
    ('CZE', 2019, 2021),  # Czechia
    ('FIN', 2016, 2021),  # Finland
    ('GRC', 2017, 2019),  # Greece
    ('HUN', 2018, 2021),  # Hungary
    ('IMN', 2017, 2020),  # Isle of Man
    ('IRL', 2016, 2021),  # Ireland
    ('KOR', 2016, 2021),  # Korea
    ('MAC', 2019, 2021),  # Macau
    ('MUS', 2019, 2021),  # Mauritius
    ('MAR', 2021, 2021),  # Morocco
    ('NLD', 2016, 2017),  # Netherlands
    ('NOR', 2016, 2017),  # Norway
    ('NZL', 2018, 2021),  # New Zealand
    ('POL', 2019, 2021),  # Poland
    ('SWE', 2016, 2021),  # Sweden
    ('GBR', 2017, 2021),  # UK
]

def get_excluded_parents_for_year(year):
    return {iso for iso, start, end in EXCLUSION_CONDITIONS if start <= year <= end}

# Load main dataset
cbcr_full = pd.read_csv(f'{data_final}/cbcr_main_no_imputation_allsubgroupsonly.csv')

# Partner info columns
partner_info_cols = ['iso_partner', 'partner_jurisdiction', 'etr_average_corrected', 'cit',
                     'tax_revenue_current_usd', 'gvt_health_expenditure', 'region_tjn',
                     'ukt', 'oecd', 'oecd_oct', 'nld_oct']

print(f"Loaded {len(cbcr_full)} rows")
print(f"Years: {sorted(cbcr_full['year'].unique())}")

Loaded 17874 rows
Years: [np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]


## 3. METHOD 1: Original Approach

This matches the per-year notebooks exactly:
1. Run `calculate_misalignment()` on good reporters
2. Calculate shares from the **result** (after negative values set to 0)
3. Sum ALL bad reporter data (domestic + foreign) and distribute using shares
4. Combine good reporters (raw) + distributed and run final misalignment

In [7]:
def run_original_method(year):
    """Original method matching per-year notebooks."""
    print(f"\n{'='*60}\nOriginal Method - Year {year}\n{'='*60}")
    
    excluded_parents = get_excluded_parents_for_year(year)
    
    # Step 1: Get good reporters (exclude non-countries and bad reporters)
    good_reporters = cbcr_full[
        (cbcr_full['year'] == year) & 
        (~cbcr_full['iso_partner'].isin(non_countries)) &
        (~cbcr_full['iso_parent'].isin(excluded_parents))
    ].copy()
    
    if good_reporters.empty:
        return None, None, None
    
    # Step 2: Run misalignment on good reporters FIRST (this sets negative values to 0)
    misalignment_good = calculate_misalignment(good_reporters.copy(), etr_max=1, weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0])
    
    # Step 3: Calculate shares from misalignment result (after cleaning)
    misalignment_for_shares = misalignment_good[['iso_parent', 'iso_partner', 'year'] + CBCR_VARS].copy()
    
    # Sum by iso_partner
    partner_totals = misalignment_for_shares.groupby('iso_partner')[CBCR_VARS].sum()
    global_totals = partner_totals.sum()
    shares = (partner_totals / global_totals).add_prefix('share_').reset_index()
    
    # Get list of iso_partners from good reporters
    iso_partners_list = good_reporters['iso_partner'].unique()
    
    # Step 4: Get bad reporters' data and sum ALL by iso_parent
    bad_reporters = cbcr_full[
        (cbcr_full['year'] == year) & 
        (cbcr_full['iso_parent'].isin(excluded_parents))
    ].copy()
    
    bad_totals = bad_reporters.groupby('iso_parent')[CBCR_VARS].sum().reset_index()
    bad_totals['year'] = year
    
    # Step 5: Distribute bad reporters using shares
    distributed_rows = []
    for _, row in bad_totals.iterrows():
        for iso_partner in iso_partners_list:
            share_row = shares[shares['iso_partner'] == iso_partner]
            if share_row.empty:
                continue
            new_row = {'iso_parent': row['iso_parent'], 'iso_partner': iso_partner, 'year': year}
            for var in CBCR_VARS:
                share_val = share_row[f'share_{var}'].values[0]
                new_row[var] = row[var] * share_val if pd.notna(share_val) else 0
            distributed_rows.append(new_row)
    
    distributed = pd.DataFrame(distributed_rows)
    
    # Step 6: Combine good reporters (RAW) with distributed bad reporters
    combined = pd.concat([good_reporters, distributed], ignore_index=True)
    
    # Step 7: Run final misalignment on combined data
    final_misalignment = calculate_misalignment(combined, etr_max=1, weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0])
    final_misalignment['year'] = year
    
    # Get unique partner info for aggregation
    available_cols = [c for c in partner_info_cols if c in final_misalignment.columns]
    unique_partners = final_misalignment.drop_duplicates(subset=['iso_partner'])[available_cols]
    
    # Aggregate results
    country_results = aggregate_country_results(final_misalignment, unique_partners, year)
    
    # Calculate bilateral
    bilateral = calculate_bilateral_by_parent(final_misalignment, year)
    
    # Print summary
    total_pos = country_results['positive_misalignment'].sum()
    total_loss = country_results['tax_revenue_loss'].sum()
    print(f"  Shifted: {total_pos:,.0f}M USD, Tax Loss: {total_loss:,.0f}M USD")
    
    return final_misalignment, country_results, bilateral

## 4. METHOD 2: Corrected Approach

Improvements:
1. Preserve domestic data (iso_partner == iso_parent) as-is from bad reporters
2. Only distribute FOREIGN aggregates
3. Calculate shares from FOREIGN data only (exclude domestic from good reporters)

In [8]:
def run_corrected_method(year):
    """Corrected method: preserves domestic, only distributes foreign."""
    print(f"\n{'='*60}\nCorrected Method - Year {year}\n{'='*60}")
    
    excluded_parents = get_excluded_parents_for_year(year)
    
    # Step 1: Get good reporters (exclude non-countries and bad reporters)
    good_reporters = cbcr_full[
        (cbcr_full['year'] == year) & 
        (~cbcr_full['iso_partner'].isin(non_countries)) &
        (~cbcr_full['iso_parent'].isin(excluded_parents))
    ].copy()
    
    if good_reporters.empty:
        return None, None, None
    
    # Step 2: Calculate shares from FOREIGN data only (iso_partner != iso_parent)
    good_foreign = good_reporters[good_reporters['iso_partner'] != good_reporters['iso_parent']].copy()
    
    partner_totals = good_foreign.groupby('iso_partner')[CBCR_VARS].sum()
    global_totals = partner_totals.sum()
    shares = (partner_totals / global_totals).add_prefix('share_').reset_index()
    
    # Get list of foreign iso_partners
    iso_partners_list = good_foreign['iso_partner'].unique()
    
    # Step 3: Get bad reporters' data
    bad_reporters = cbcr_full[
        (cbcr_full['year'] == year) & 
        (cbcr_full['iso_parent'].isin(excluded_parents))
    ].copy()
    
    # Step 4: Separate domestic from foreign for bad reporters
    # Domestic: iso_partner == iso_parent (keep as-is)
    bad_domestic = bad_reporters[bad_reporters['iso_partner'] == bad_reporters['iso_parent']].copy()
    
    # Foreign: iso_partner != iso_parent (sum and distribute)
    bad_foreign = bad_reporters[bad_reporters['iso_partner'] != bad_reporters['iso_parent']].copy()
    bad_foreign_totals = bad_foreign.groupby('iso_parent')[CBCR_VARS].sum().reset_index()
    bad_foreign_totals['year'] = year
    
    print(f"  Bad reporters: {len(excluded_parents)} countries")
    print(f"  - Domestic rows preserved: {len(bad_domestic)}")
    print(f"  - Foreign aggregates to distribute: {len(bad_foreign_totals)}")
    
    # Step 5: Distribute ONLY foreign aggregates
    distributed_rows = []
    for _, row in bad_foreign_totals.iterrows():
        iso_parent = row['iso_parent']
        for iso_partner in iso_partners_list:
            # Skip if this would be domestic
            if iso_partner == iso_parent:
                continue
            share_row = shares[shares['iso_partner'] == iso_partner]
            if share_row.empty:
                continue
            new_row = {'iso_parent': iso_parent, 'iso_partner': iso_partner, 'year': year}
            for var in CBCR_VARS:
                share_val = share_row[f'share_{var}'].values[0]
                new_row[var] = row[var] * share_val if pd.notna(share_val) else 0
            distributed_rows.append(new_row)
    
    distributed = pd.DataFrame(distributed_rows)
    
    # Step 6: Combine: good reporters + bad domestic (preserved) + distributed foreign
    combined = pd.concat([good_reporters, bad_domestic, distributed], ignore_index=True)
    
    # Filter out any remaining non-country partners
    combined = combined[~combined['iso_partner'].isin(non_countries)]
    
    # Step 7: Run final misalignment
    final_misalignment = calculate_misalignment(combined, etr_max=1, weights=[1/6, 1/3, 1/3, 1/6, 0, 0, 0, 0])
    final_misalignment['year'] = year
    
    # Get unique partner info
    available_cols = [c for c in partner_info_cols if c in final_misalignment.columns]
    unique_partners = final_misalignment.drop_duplicates(subset=['iso_partner'])[available_cols]
    
    # Aggregate results
    country_results = aggregate_country_results(final_misalignment, unique_partners, year)
    
    # Calculate bilateral
    bilateral = calculate_bilateral_by_parent(final_misalignment, year)
    
    # Print summary
    total_pos = country_results['positive_misalignment'].sum()
    total_loss = country_results['tax_revenue_loss'].sum()
    print(f"  Shifted: {total_pos:,.0f}M USD, Tax Loss: {total_loss:,.0f}M USD")
    
    return final_misalignment, country_results, bilateral

## 5. Run Both Methods for All Years

In [ ]:
# Store results
results_original = {'country': [], 'bilateral': [], 'aggregate': []}
results_corrected = {'country': [], 'bilateral': [], 'aggregate': []}

for year in range(first_year, first_year + n_years):
    # Original method
    mis_orig, country_orig, bilateral_orig = run_original_method(year)
    if country_orig is not None:
        results_original['country'].append(country_orig)
        results_original['bilateral'].append(bilateral_orig)
        total_pos = country_orig['positive_misalignment'].sum()
        total_loss = country_orig['tax_revenue_loss'].sum()
        results_original['aggregate'].append({
            'year': year, 'method': 'original',
            'total_shifted_musd': total_pos,
            'total_tax_loss_musd': total_loss
        })
        country_orig.to_csv(output_base / 'method_original' / f'country_results_{year}.csv', index=False)
        bilateral_orig.to_csv(output_base / 'method_original' / f'bilateral_{year}.csv', index=False)
    
    # Corrected method
    mis_corr, country_corr, bilateral_corr = run_corrected_method(year)
    if country_corr is not None:
        results_corrected['country'].append(country_corr)
        results_corrected['bilateral'].append(bilateral_corr)
        total_pos = country_corr['positive_misalignment'].sum()
        total_loss = country_corr['tax_revenue_loss'].sum()
        results_corrected['aggregate'].append({
            'year': year, 'method': 'corrected',
            'total_shifted_musd': total_pos,
            'total_tax_loss_musd': total_loss
        })
        country_corr.to_csv(output_base / 'method_corrected' / f'country_results_{year}.csv', index=False)
        bilateral_corr.to_csv(output_base / 'method_corrected' / f'bilateral_{year}.csv', index=False)

# Combine all years
if results_original['country']:
    pd.concat(results_original['country']).to_csv(output_base / 'method_original' / 'country_results_all_years.csv', index=False)
    pd.concat(results_original['bilateral']).to_csv(output_base / 'method_original' / 'bilateral_all_years.csv', index=False)

if results_corrected['country']:
    pd.concat(results_corrected['country']).to_csv(output_base / 'method_corrected' / 'country_results_all_years.csv', index=False)
    pd.concat(results_corrected['bilateral']).to_csv(output_base / 'method_corrected' / 'bilateral_all_years.csv', index=False)


Original Method - Year 2016


## 6. Compare Results Between Methods

In [ ]:
# Create comparison table
agg_orig = pd.DataFrame(results_original['aggregate'])
agg_corr = pd.DataFrame(results_corrected['aggregate'])

comparison = agg_orig.merge(agg_corr, on='year', suffixes=('_original', '_corrected'))
comparison['diff_shifted_musd'] = comparison['total_shifted_musd_corrected'] - comparison['total_shifted_musd_original']
comparison['diff_shifted_pct'] = 100 * comparison['diff_shifted_musd'] / comparison['total_shifted_musd_original']
comparison['diff_loss_musd'] = comparison['total_tax_loss_musd_corrected'] - comparison['total_tax_loss_musd_original']
comparison['diff_loss_pct'] = 100 * comparison['diff_loss_musd'] / comparison['total_tax_loss_musd_original']

comparison.to_csv(output_base / 'comparison' / 'method_comparison.csv', index=False)

print("\n" + "="*80)
print("COMPARISON: Original vs Corrected Method")
print("="*80)
print("\nAggregate Results (in million USD):")
display_cols = ['year', 'total_shifted_musd_original', 'total_shifted_musd_corrected', 
                'diff_shifted_pct', 'total_tax_loss_musd_original', 'total_tax_loss_musd_corrected',
                'diff_loss_pct']
print(comparison[display_cols].to_string(index=False))

print(f"\nAverage difference in shifted profits: {comparison['diff_shifted_pct'].mean():.2f}%")
print(f"Average difference in tax loss: {comparison['diff_loss_pct'].mean():.2f}%")


COMPARISON: Original vs Corrected Method

Aggregate Results (in million USD):
 year  total_shifted_musd_original  total_shifted_musd_corrected  diff_shifted_pct  total_tax_loss_musd_original  total_tax_loss_musd_corrected  diff_loss_pct
 2016                   679,640.57                    667,613.59             -1.77                    172,973.16                     167,420.09          -3.21
 2017                 1,139,568.19                  1,131,422.86             -0.71                    292,587.04                     286,827.04          -1.97
 2018                 1,152,390.36                  1,196,169.89              3.80                    293,945.44                     287,373.70          -2.24
 2019                 1,264,135.97                  1,279,265.77              1.20                    323,224.43                     315,610.83          -2.36
 2020                 1,365,649.89                  1,660,423.04             21.58                    329,699.56              

In [ ]:
# Compare country-level results for most recent year
if results_original['aggregate'] and results_corrected['aggregate']:
    latest_year = max(r['year'] for r in results_original['aggregate'])

    orig_latest = [df for df in results_original['country'] if df['year'].iloc[0] == latest_year][0]
    corr_latest = [df for df in results_corrected['country'] if df['year'].iloc[0] == latest_year][0]

    country_comparison = orig_latest[['iso_partner', 'positive_misalignment', 'negative_misalignment', 'tax_revenue_loss']].merge(
        corr_latest[['iso_partner', 'positive_misalignment', 'negative_misalignment', 'tax_revenue_loss']],
        on='iso_partner', suffixes=('_orig', '_corr')
    )

    country_comparison['diff_loss'] = country_comparison['tax_revenue_loss_corr'] - country_comparison['tax_revenue_loss_orig']
    country_comparison['diff_loss_pct'] = 100 * country_comparison['diff_loss'] / country_comparison['tax_revenue_loss_orig'].replace(0, np.nan)

    print(f"\nCountry-level differences ({latest_year}) - Top 10 by absolute difference:")
    top10 = country_comparison.reindex(country_comparison['diff_loss'].abs().sort_values(ascending=False).index).head(10)
    print(top10[['iso_partner', 'tax_revenue_loss_orig', 'tax_revenue_loss_corr', 'diff_loss', 'diff_loss_pct']].to_string(index=False))

    country_comparison.to_csv(output_base / 'comparison' / f'country_comparison_{latest_year}.csv', index=False)


Country-level differences (2022) - Top 10 by absolute difference:
iso_partner  tax_revenue_loss_orig  tax_revenue_loss_corr  diff_loss  diff_loss_pct
        ABW                   0.48                   0.48       0.00           0.00
        AFG                   0.97                   0.97       0.00           0.00
        AGO                   1.70                   1.70       0.00           0.00
        AIA                   0.00                   0.00       0.00            NaN
        ALB                   4.96                   4.96       0.00           0.00
        AND                   0.00                   0.00       0.00            NaN
        ARE                   0.00                   0.00       0.00            NaN
        ARG               4,838.27               4,838.27       0.00           0.00
        ARM                   7.08                   7.08       0.00           0.00
        ATG                  -0.00                  -0.00       0.00            NaN


## 7. Aggregate Bilateral Results

In [ ]:
# Aggregate bilateral across all parents for corrected method
if results_corrected['bilateral']:
    bilateral_all = pd.concat(results_corrected['bilateral'])
    
    bilateral_agg = bilateral_all.groupby(['year', 'iso_responsible', 'iso_affected']).agg(
        total_shifted_musd=('shifted_profit_musd', 'sum'),
        total_tax_loss_musd=('tax_loss_musd', 'sum'),
        n_parents=('iso_parent', 'nunique')
    ).reset_index()
    
    bilateral_agg.to_csv(output_base / 'bilateral' / 'bilateral_aggregated_corrected.csv', index=False)
    
    # Also save to TJN shared folder for IFF portal
    tjn_bilateral_path = Path(tjn_shared_bilateral)
    if tjn_bilateral_path.exists():
        bilateral_agg.to_csv(tjn_bilateral_path / 'corporate_taxabuse_iffportal.csv', index=False)
        print(f"Saved: {tjn_bilateral_path / 'corporate_taxabuse_iffportal.csv'}")
    
    # Top pairs for latest year
    latest_year = bilateral_agg['year'].max()
    latest = bilateral_agg[bilateral_agg['year'] == latest_year]
    print(f"\nTop 15 bilateral pairs by tax loss ({latest_year}, corrected method):")
    print(latest.nlargest(15, 'total_tax_loss_musd')[['iso_responsible', 'iso_affected', 'total_tax_loss_musd']].to_string(index=False))

Saved: C:\Users\aliso\Tax Justice Network Ltd\TJN - Shared Documents\Research team\Projects long-term\SOTJ\Tables\bilateral\corporate_taxabuse_iffportal.csv

Top 15 bilateral pairs by tax loss (2022, corrected method):
iso_responsible iso_affected  total_tax_loss_musd
            CAN          USA            12,203.05
            IRL          USA            12,131.81
            SGP          USA             9,500.44
            CHE          USA             6,381.39
            CHN          HKG             5,280.88
            IRL          GBR             3,673.05
            BRA          ARG             3,366.73
            PRI          USA             3,011.36
            SGP          GBR             2,961.09
            JPN          GBR             2,834.04
            GBR          FRA             2,806.01
            HKG          USA             2,674.92
            HKG          CHN             2,641.32
            TWN          CHN             2,581.88
            USA          FRA   

In [ ]:
# Summary by tax haven (responsible)
if results_corrected['bilateral']:
    by_responsible = bilateral_agg.groupby(['year', 'iso_responsible']).agg(
        total_tax_loss_caused_musd=('total_tax_loss_musd', 'sum'),
        n_affected=('iso_affected', 'nunique')
    ).reset_index().sort_values(['year', 'total_tax_loss_caused_musd'], ascending=[True, False])
    
    by_responsible.to_csv(output_base / 'bilateral' / 'summary_by_responsible_corrected.csv', index=False)
    
    print(f"\nTop 15 tax havens by harm caused ({latest_year}):")
    print(by_responsible[by_responsible['year'] == latest_year].head(15).to_string(index=False))


Top 15 tax havens by harm caused (2022):
 year iso_responsible  total_tax_loss_caused_musd  n_affected
 2022             IRL                   24,643.00         172
 2022             SGP                   21,283.48         166
 2022             CHN                   15,641.66         181
 2022             CAN                   15,513.75         170
 2022             HKG                   15,303.28         180
 2022             CHE                   15,177.88         164
 2022             BRA                   12,490.38         167
 2022             SAU                   10,305.09         141
 2022             JPN                    9,512.65         147
 2022             NOR                    9,220.91         167
 2022             MEX                    8,822.09         152
 2022             DEU                    7,353.09         139
 2022             PRI                    5,887.40         165
 2022             NLD                    5,145.26         183
 2022             AUS       

In [ ]:
# Summary by affected country
if results_corrected['bilateral']:
    by_affected = bilateral_agg.groupby(['year', 'iso_affected']).agg(
        total_tax_loss_suffered_musd=('total_tax_loss_musd', 'sum'),
        n_responsible=('iso_responsible', 'nunique')
    ).reset_index().sort_values(['year', 'total_tax_loss_suffered_musd'], ascending=[True, False])
    
    by_affected.to_csv(output_base / 'bilateral' / 'summary_by_affected_corrected.csv', index=False)
    
    print(f"\nTop 15 countries by harm suffered ({latest_year}):")
    print(by_affected[by_affected['year'] == latest_year].head(15).to_string(index=False))


Top 15 countries by harm suffered (2022):
 year iso_affected  total_tax_loss_suffered_musd  n_responsible
 2022          USA                     69,594.44            173
 2022          GBR                     28,590.88            173
 2022          FRA                     17,333.59            170
 2022          CHN                     10,262.75            126
 2022          HKG                      8,164.68            132
 2022          SGP                      7,861.22            148
 2022          DEU                      7,758.47            171
 2022          CYM                      7,386.97            146
 2022          ESP                      6,846.13            170
 2022          IND                      5,970.92            167
 2022          LUX                      5,292.12            151
 2022          ARG                      4,697.27            153
 2022          ITA                      4,427.91            172
 2022          MEX                      4,281.73            1

## 8. Summary

In [ ]:
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print("\nFiles saved:")
print(f"  - Original method: {output_base / 'method_original'}")
print(f"  - Corrected method: {output_base / 'method_corrected'}")
print(f"  - Bilateral results: {output_base / 'bilateral'}")
print(f"  - Comparison: {output_base / 'comparison'}")

print("\nKey differences between methods:")
print("  ORIGINAL METHOD:")
print("    - Calculates shares AFTER running initial misalignment (negative values set to 0)")
print("    - Distributes ALL bad reporter data (including domestic)")
print("    - Matches results from per-year notebooks (3.1, 3.2, etc.)")
print("")
print("  CORRECTED METHOD:")
print("    - Preserves domestic data (iso_partner == iso_parent) as-is")
print("    - Only distributes FOREIGN aggregates")
print("    - Calculates shares from FOREIGN data only")
print("    - More accurate: domestic operations should not be redistributed")


SUMMARY

Files saved:
  - Original method: ..\output\tables\method_original
  - Corrected method: ..\output\tables\method_corrected
  - Bilateral results: ..\output\tables\bilateral
  - Comparison: ..\output\tables\comparison

Key differences between methods:
  ORIGINAL METHOD:
    - Calculates shares AFTER running initial misalignment (negative values set to 0)
    - Distributes ALL bad reporter data (including domestic)
    - Matches results from per-year notebooks (3.1, 3.2, etc.)

  CORRECTED METHOD:
    - Preserves domestic data (iso_partner == iso_parent) as-is
    - Only distributes FOREIGN aggregates
    - Calculates shares from FOREIGN data only
    - More accurate: domestic operations should not be redistributed
